# 4. Topological "Wave" Scheduling
**Difficulty:** 🟡 Medium · **Topic:** Graphs / Topological Sort · **Pattern:** Kahn's grouped by level (cf. LeetCode 210)

> **DevRev context:** a workflow is a graph of tasks with dependencies (a **DAG**). To run it as fast as possible you want to execute, in each round, **every task whose dependencies are already done** — a **parallel-safe "wave"** — then the next wave, and so on. This is topological sort, but grouped into levels. If the graph has a cycle, it can never be scheduled.

## 💡 Concepts

**Core concept(s):** **Kahn's algorithm**, but instead of emitting one node at a time, emit **all currently-ready nodes** as one wave.

**Why it applies here:** A task is ready when its in-degree (remaining prerequisites) hits 0. Every ready task in a round is independent of the others, so they can all run in parallel. Removing a whole wave unlocks the next wave. The number of waves is the length of the **critical path** — the minimum number of sequential rounds.

**Key intuition:** Run everything with zero remaining prerequisites together; finishing a wave lowers its dependents' counts, exposing the next wave.

---

### 📚 What is a DAG / in-degree?
A **DAG** is a Directed Acyclic Graph — arrows, no loops (a valid task-dependency graph). A node's **in-degree** is how many arrows point *into* it = how many prerequisites it still has. In-degree 0 means "ready to run now".

### 📚 What is Topological Sort (Kahn's algorithm)?
For a directed graph with **no cycles (a DAG)**, a topological order lists nodes so every arrow points forward (dependencies first). **Kahn's method:** repeatedly take nodes with **0 remaining prerequisites** (in-degree 0), output them, and remove their outgoing edges. If some node never reaches in-degree 0, there is a **cycle**.

---

**Prerequisite knowledge:**
- In-degree counting.
- A queue/frontier of ready nodes.
- Cycle detection (leftover nodes = a cycle).

## 📝 Problem

Given `n` tasks (`0..n-1`) and a list of dependencies `[a, b]` meaning **b must finish before a**,
return the tasks grouped into **waves**: wave 0 is every task with no prerequisites, wave 1 is every task
whose prerequisites are all in wave 0, and so on. If the tasks can't all be scheduled (a **cycle**), return `None`.

**Example**
```
n = 6, deps = [[2,0],[3,0],[4,1],[4,2],[5,3],[5,4]]
0 and 1 have no prereqs -> wave 0
2,3 depend only on {0} -> wave 1
4 depends on {1,2}, done after wave 1 -> wave 2
5 depends on {3,4} -> wave 3
-> [[0,1],[2,3],[4],[5]]
```

> Two approaches: a naive re-scan-each-wave `O(V·E)` and Kahn's-by-level `O(V+E)`.

### Approach 1 — Re-scan Dependencies Each Wave (worst)

**Idea:** Each round, recompute every remaining task's prerequisite count by scanning *all* dependencies,
take the ready ones as a wave, remove them, and repeat.

**Time:** `O(V·E)` — a full `O(E)` scan for up to `V` waves. **Space:** `O(V+E)`.

In [ ]:
from typing import List, Optional

def schedule_waves_naive(n: int, deps: List[List[int]]) -> Optional[List[List[int]]]:
    remaining = set(range(n))                  # tasks not yet scheduled
    waves = []
    while remaining:
        # Recompute how many prerequisites each remaining task still has (naive: scan all deps).
        indeg = {x: 0 for x in remaining}
        for a, b in deps:
            if a in remaining and b in remaining:
                indeg[a] += 1                  # a still waits on b
        ready = [x for x in remaining if indeg[x] == 0]   # runnable this round
        if not ready:
            return None                        # nothing ready but tasks remain -> a cycle
        waves.append(sorted(ready))
        remaining -= set(ready)                # this wave is done
    return waves

### Approach 2 — Kahn's by Level (optimal)

**Idea:** Compute every in-degree once. The frontier of in-degree-0 tasks is a wave; process the whole wave,
decrementing each dependent's count, and collect the newly-freed tasks as the next wave.

**Time:** `O(V + E)`. **Space:** `O(V + E)`.

In [ ]:
from typing import List, Optional
from collections import defaultdict

def schedule_waves_kahn(n: int, deps: List[List[int]]) -> Optional[List[List[int]]]:
    graph = defaultdict(list)                  # b -> [tasks that depend on b]
    indeg = [0] * n                            # remaining prerequisites per task
    for a, b in deps:
        graph[b].append(a)                     # finishing b unlocks a
        indeg[a] += 1
    wave = [x for x in range(n) if indeg[x] == 0]   # everything ready to start
    waves, scheduled = [], 0
    while wave:
        waves.append(sorted(wave))             # this whole wave runs in parallel
        nxt = []
        for task in wave:
            scheduled += 1
            for dependent in graph[task]:      # this task finishing frees its dependents
                indeg[dependent] -= 1
                if indeg[dependent] == 0:      # all prereqs met -> ready next wave
                    nxt.append(dependent)
        wave = nxt
    return waves if scheduled == n else None   # didn't schedule everything -> a cycle

In [ ]:
# Correctness check
n, deps = 6, [[2, 0], [3, 0], [4, 1], [4, 2], [5, 3], [5, 4]]
expected = [[0, 1], [2, 3], [4], [5]]
a, b = schedule_waves_naive(n, deps), schedule_waves_kahn(n, deps)
print("naive:", a)
print("kahn :", b)
assert a == b == expected, "mismatch!"

# a cycle cannot be scheduled
assert schedule_waves_naive(2, [[0, 1], [1, 0]]) is None
assert schedule_waves_kahn(2, [[0, 1], [1, 0]]) is None

# no dependencies -> everything runs in a single wave
assert schedule_waves_kahn(3, []) == [[0, 1, 2]]
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

We time each approach on inputs of growing size `n` and read the **doubling ratio**.

| Theoretical | Ratio `n`→`2n` |
|---|---|
| `O(n)` / `O(V+E)` | ≈ **2×** |
| `O(n log n)`      | ≈ **2×** (slightly more) |
| `O(n²)`           | ≈ **4×** |

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")): break
    _root = os.path.dirname(_root)
if _root not in sys.path: sys.path.insert(0, _root)
from bench_utils import benchmark

def make_worst_case(n):
    # A long dependency chain (0<-1<-2<-...): n waves of 1 task each.
    # The naive version rescans all deps every wave -> O(V*E).
    deps = [[i, i - 1] for i in range(1, n)]
    return (n, deps)
solutions = {
    "re-scan  O(V*E)": schedule_waves_naive,
    "kahn     O(V+E)": schedule_waves_kahn,
}
sizes = [200, 400, 800, 1600]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Kahn's by level = parallel batches:** grouping the ready frontier gives the *minimum number of sequential rounds* (the critical-path length) and the set of tasks safe to run together.
- **Decrement, don't re-scan:** update in-degrees as edges are removed (O(V+E)) instead of recomputing them each round (O(V·E)).
- **Leftover nodes = a cycle:** if you can't schedule all V tasks, the graph has a dependency loop.
- **Signal:** "run tasks in dependency order", "which can run in parallel", "detect a scheduling cycle".
- **DevRev / related:** workflow/pipeline orchestration, build systems, Course Schedule II (LeetCode 210), Alien Dictionary.
- **Common pitfalls:** (1) mixing up edge direction (`a` depends on `b` vs `b` unlocks `a`); (2) forgetting the cycle check; (3) emitting one node at a time when the question wants parallel waves.